
# Zener viscoelastic model


In [ ]:
import os
import numpy as np
import matplotlib.pyplot as plt
import simcoon as sim

plt.rcParams["figure.figsize"] = (18, 10)

The Poynting-Thomson (Zener) constitutive law is a rate-dependent, isotropic,
linear viscoelastic model that accounts for thermal strains. It consists of
an elastic spring in parallel with a Maxwell element (spring + dashpot in series).

Seven parameters are required:

1. The thermoelastic Young's modulus $E_0$
2. The thermoelastic Poisson's ratio $\nu_0$
3. The coefficient of thermal expansion $\alpha$
4. The viscoelastic Young's modulus of the Zener branch $E_1$
5. The viscoelastic Poisson's ratio of the Zener branch $\nu_1$
6. The bulk viscosity of the Zener branch $\eta_B$
7. The shear viscosity of the Zener branch $\eta_S$

The viscoelastic material constitutive law is implemented using a
*fast scalar updating method*. The updated stress is provided for 1D,
plane stress, and generalized plane strain/3D analysis.



In [ ]:
umat_name = "ZENER"  # 5 character code for the Zener model
nstatev = 8  # Number of internal variables

# Material parameters
E_0 = 3000.0  # Thermoelastic Young's modulus (MPa)
nu_0 = 0.4  # Thermoelastic Poisson's ratio
alpha = 0.0  # Thermal expansion coefficient
E_1 = 100.0  # Viscoelastic Young's modulus (MPa)
nu_1 = 0.3  # Viscoelastic Poisson's ratio
eta_S = 4000.0  # Shear viscosity
eta_B = eta_S / 4.0  # Bulk viscosity

psi_rve = 0.0
theta_rve = 0.0
phi_rve = 0.0
solver_type = 0
corate_type = 1

props = np.array([E_0, nu_0, alpha, E_1, nu_1, eta_B, eta_S])

path_data = "../data"
pathfile = "ZENER_path.json"

The loading path is read in Python and the simulation runs in memory: no result
file is written, and the histories come back as component-first arrays.



In [ ]:
blocks, T_init, _ = sim.solver.load_path_json(os.path.join(path_data, pathfile))

res = sim.solver.solve(
    blocks,
    umat_name,
    props,
    nstatev,
    T_init=T_init,
    solver_type=solver_type,
    corate=corate_type,
    orientation=(psi_rve, theta_rve, phi_rve),
)

## Plotting the results

We plot the stress-strain response which exhibits the characteristic
rate-dependent behavior of the Zener viscoelastic model.



In [ ]:
e11, e22, e33, e12, e13, e23 = res["Strain"]
s11, s22, s33, s12, s13, s23 = res["Stress"]
time, T = res["Time"], res["Temp"]
Wm, Wm_r, Wm_ir, Wm_d = res["Wm"]

fig = plt.figure()

# First subplot: Stress vs Strain
ax1 = fig.add_subplot(1, 2, 1)
plt.grid(True)
plt.tick_params(axis="both", which="major", labelsize=15)
plt.xlabel(r"Strain $\varepsilon_{11}$", size=15)
plt.ylabel(r"Stress $\sigma_{11}$ (MPa)", size=15)
plt.plot(e11, s11, c="blue", label="Zener model")
plt.legend(loc="best")

# Second subplot: Work terms vs Time
ax2 = fig.add_subplot(1, 2, 2)
plt.grid(True)
plt.tick_params(axis="both", which="major", labelsize=15)
plt.xlabel("time (s)", size=15)
plt.ylabel(r"$W_m$", size=15)
plt.plot(time, Wm, c="black", label=r"$W_m$")
plt.plot(time, Wm_r, c="green", label=r"$W_m^r$")
plt.plot(time, Wm_ir, c="blue", label=r"$W_m^{ir}$")
plt.plot(time, Wm_d, c="red", label=r"$W_m^d$")
plt.legend(loc="best")

plt.show()